# Optimize Weld's Experiment with Ax


In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from ax.api.client import Client
from ax.api.configs import ChoiceParameterConfig, RangeParameterConfig


In [73]:
# Bootstrap MSVC toolchain inside the kernel process so BoTorch can build fused kernels on Windows.


def _load_msvc_env_from_vcvars() -> bool:
    vcvars = Path(
        r"C:\Program Files (x86)\Microsoft Visual Studio\2022\BuildTools\VC\Auxiliary\Build\vcvars64.bat"
    )
    if not vcvars.exists():
        print(f"MSVC bootstrap skipped: {vcvars} not found.")
        return False

    dump_env_cmd = Path(tempfile.gettempdir()) / "_dump_msvc_env.cmd"
    dump_env_cmd.write_text(
        f'@echo off\ncall "{vcvars}" >nul\nif errorlevel 1 exit /b 1\nset\n',
        encoding="utf-8",
    )

    try:
        proc = subprocess.run(
            ["cmd.exe", "/d", "/c", str(dump_env_cmd)],
            capture_output=True,
            text=True,
            encoding="mbcs",
            errors="replace",
            check=True,
        )
    except subprocess.CalledProcessError as exc:
        print(f"MSVC bootstrap failed with return code {exc.returncode}.")
        if exc.stderr:
            print(exc.stderr.strip())
        return False

    for line in proc.stdout.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value

    # Ensure the kernel keeps the active venv Scripts folder on PATH.
    scripts_dir = str(Path(sys.executable).resolve().parent)
    path_parts = os.environ.get("PATH", "").split(os.pathsep)
    if scripts_dir not in path_parts:
        os.environ["PATH"] = scripts_dir + os.pathsep + os.environ.get("PATH", "")

    return True


loaded = _load_msvc_env_from_vcvars()
print("MSVC env loaded:", loaded)
print("cl:", shutil.which("cl"))
print("ninja:", shutil.which("ninja"))

MSVC bootstrap failed with return code 255.
The input line is too long.
The syntax of the command is incorrect.
MSVC env loaded: False
cl: C:\Program Files (x86)\Microsoft Visual Studio\2022\BuildTools\VC\Tools\MSVC\14.44.35207\bin\HostX64\x64\cl.EXE
ninja: c:\Users\admin\Coding\research\weld-ml-v2\.venv\Scripts\ninja.EXE


In [74]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig


In [75]:
NUM_NEW_EXP = 5

In [76]:
# 1. Initialize the Client.
client = Client(random_seed=42)

In [77]:
folder_path = Path(os.getcwd()) / "data"

In [78]:
exp = pd.read_excel(folder_path / "experiments.xlsx")
display(exp)


,trial_index,R,W,D,S,P,O,visual_quality,sigma_peak,uts
0,0,1700,70,20,round,0.105000,O2,1,127.755842,256.639476
1,1,1462,81,25,square,0.162470,O1,1,212.418007,228.246757
2,2,1733,44,18,round,0.102709,O2,0,124.219927,256.303258
3,3,1997,92,21,round,0.136314,O2,0,172.034853,232.938675
4,4,1649,63,13,square,0.055458,O1,0,131.353171,252.499414
5,5,1462,81,25,square,0.162470,O1,1,203.237882,217.516872
6,6,1687,73,19,round,0.079064,O2,1,137.265759,265.346123
7,7,1687,73,19,round,0.133737,O2,1,128.850675,257.957394
8,8,1435,70,14,round,0.104998,O2,1,211.114702,211.308453
9,9,1718,71,11,round,0.090116,O2,1,117.897305,259.486250


In [79]:
print(exp.columns)
cols_output = [
    "visual_quality",
    "sigma_peak",
    "uts",
]

Index(['trial_index', 'R', 'W', 'D', 'S', 'P', 'O', 'visual_quality',
       'sigma_peak', 'uts'],
      dtype='str')


In [80]:
filt_removed = exp[cols_output].isna().all(axis=1)

In [81]:
exp_todo = exp[filt_removed].copy()
exp = exp[~filt_removed].copy()
display(exp)
display(exp_todo)

,trial_index,R,W,D,S,P,O,visual_quality,sigma_peak,uts
0,0,1700,70,20,round,0.105000,O2,1,127.755842,256.639476
1,1,1462,81,25,square,0.162470,O1,1,212.418007,228.246757
2,2,1733,44,18,round,0.102709,O2,0,124.219927,256.303258
3,3,1997,92,21,round,0.136314,O2,0,172.034853,232.938675
4,4,1649,63,13,square,0.055458,O1,0,131.353171,252.499414
5,5,1462,81,25,square,0.162470,O1,1,203.237882,217.516872
6,6,1687,73,19,round,0.079064,O2,1,137.265759,265.346123
7,7,1687,73,19,round,0.133737,O2,1,128.850675,257.957394
8,8,1435,70,14,round,0.104998,O2,1,211.114702,211.308453
9,9,1718,71,11,round,0.090116,O2,1,117.897305,259.486250


,trial_index,R,W,D,S,P,O,visual_quality,sigma_peak,uts


In [82]:
# 2. Configure where Ax will search.

CATEGORIES = {"S": ("square", "round"), "O": ("O1", "O2")}

bound_R = [1400, 2000]  # Rotatinal speed
bound_W = [40, 100]  # Welding speed
bound_D = [10, 30]  # Dwell time
bound_P = [0.03, 0.18]  # Plunge Depth

parameters = [
    RangeParameterConfig(
        name="R",
        bounds=bound_R,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="W",
        bounds=bound_W,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="D",
        bounds=bound_D,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="P",
        bounds=bound_P,
        parameter_type="float",
    ),
    ChoiceParameterConfig(
        name="S",
        parameter_type="str",
        values=list(CATEGORIES["S"]),
    ),
    ChoiceParameterConfig(
        name="O",
        parameter_type="str",
        values=list(CATEGORIES["O"]),
    ),
]

client.configure_experiment(
    name="parn_exp",
    parameters=parameters,
)

c:\Users\admin\Coding\research\weld-ml-v2\.venv\Lib\site-packages\ax\api\utils\instantiation\from_config.py:91: AxParameterWarning: `is_ordered` is not specified for `ChoiceParameter` "S". Defaulting to `True`  since there are exactly two choices.. To override this behavior (or avoid this warning), specify `is_ordered` during `ChoiceParameter` construction. Note that choice parameters with exactly 2 choices are always considered ordered and that the user-supplied `is_ordered` has no effect in this particular case.
  return ChoiceParameter(
c:\Users\admin\Coding\research\weld-ml-v2\.venv\Lib\site-packages\ax\api\utils\instantiation\from_config.py:91: AxParameterWarning: `is_ordered` is not specified for `ChoiceParameter` "O". Defaulting to `True`  since there are exactly two choices.. To override this behavior (or avoid this warning), specify `is_ordered` during `ChoiceParameter` construction. Note that choice parameters with exactly 2 choices are always considered ordered and that the 

In [83]:
MAX_SIGMA_PEAK = 500
MIN_UTS = 200

# The framework will maximize the objective by default, so we use negative signs to minimize the values.
# The sigma_peak is to be minimized, while the uts is to be maximized. Therefore, we use -sigma_peak and uts in the objective.
client.configure_optimization(
    objective="-sigma_peak, uts",  # TODO: double check the sign
    outcome_constraints=[
        f"sigma_peak <= {MAX_SIGMA_PEAK}",
        f"uts >= {MIN_UTS}",
        "visual_quality >= 0.5",
    ],
)

In [84]:
for idx, row in exp.iterrows():
    trial_index = client.attach_trial(
        # If Excel round-trips values as floats (e.g. 1499.9999999 due to float storage),
        # int() truncates down instead of rounding, silently shifting your attached parameter by 1.
        # Safer: int(round(row["R1"])).
        parameters={
            "R": int(round(row["R"])),
            "W": int(round(row["W"])),
            "D": int(round(row["D"])),
            "P": float(row["P"]),
            "S": row["S"],
            "O": row["O"],
        }
    )
    visual_quality = float(row["visual_quality"])
    sigma_peak = row["sigma_peak"]
    uts = row["uts"]

    print(
        f"Trial {trial_index}: visual_quality={visual_quality}, sigma_peak={sigma_peak}, uts={uts}"
    )
    client.complete_trial(
        trial_index=trial_index,
        raw_data={
            "visual_quality": visual_quality,
            "sigma_peak": sigma_peak,
            "uts": uts,
        },
    )

[INFO 10-22 09:18:14] ax.api.client: Trial 0 marked COMPLETED.
[INFO 10-22 09:18:14] ax.api.client: Trial 1 marked COMPLETED.
[INFO 10-22 09:18:14] ax.api.client: Trial 2 marked COMPLETED.
[INFO 10-22 09:18:14] ax.api.client: Trial 3 marked COMPLETED.
[INFO 10-22 09:18:14] ax.api.client: Trial 4 marked COMPLETED.
[INFO 10-22 09:18:14] ax.api.client: Trial 5 marked COMPLETED.
[INFO 10-22 09:18:14] ax.api.client: Trial 6 marked COMPLETED.


Trial 0: visual_quality=1.0, sigma_peak=127.7558417687471, uts=256.6394756835435
Trial 1: visual_quality=1.0, sigma_peak=212.4180068698516, uts=228.2467573841725
Trial 2: visual_quality=0.0, sigma_peak=124.2199270543483, uts=256.3032577925962
Trial 3: visual_quality=0.0, sigma_peak=172.0348529710814, uts=232.9386748434291
Trial 4: visual_quality=0.0, sigma_peak=131.3531707791065, uts=252.4994135383711
Trial 5: visual_quality=1.0, sigma_peak=203.2378819313057, uts=217.516871688526
Trial 6: visual_quality=1.0, sigma_peak=137.2657593692274, uts=265.3461231857516
Trial 7: visual_quality=1.0, sigma_peak=128.8506745601932, uts=257.957394354209


[INFO 10-22 09:18:14] ax.api.client: Trial 7 marked COMPLETED.
[INFO 10-22 09:18:14] ax.api.client: Trial 8 marked COMPLETED.
[INFO 10-22 09:18:14] ax.api.client: Trial 9 marked COMPLETED.


Trial 8: visual_quality=1.0, sigma_peak=211.1147021453583, uts=211.308452700761
Trial 9: visual_quality=1.0, sigma_peak=117.897304512137, uts=259.486249535395


In [85]:
if exp_todo.shape[0] == 0:
    suggest_params = []
    try:
        # Request only the intended number of new trials once.
        data = client.get_next_trials(max_trials=NUM_NEW_EXP)
        for trial_index, parameters in data.items():
            print(f"Trial {trial_index} with parameters {parameters}")
            suggest_params.append({"trial_index": trial_index, **parameters})
    except Exception as e:
        if type(e).__name__ == "DataRequiredError":
            print(
                "Ax cannot generate more trials yet: more observed trial data is "
                "required before moving past Sobol."
            )
        else:
            raise

    if suggest_params:
        _exp = pd.DataFrame(suggest_params)
        _exp[cols_output] = pd.NA
        exp = pd.concat([exp, _exp], ignore_index=True).reset_index(drop=True)
        display(exp)
        SAVE_EXCEL = True
    else:
        SAVE_EXCEL = False
else:
    print("There are still experiment to be conducted. Skip generating new sets")
    SAVE_EXCEL = False

[INFO 10-22 09:18:14] ax.api.client: GenerationStrategy(name='Center+Sobol+MBM:fast', nodes=[CenterGenerationNode(next_node_name='Sobol', use_existing_trials_for_initialization=True), GenerationNode(name='Sobol', generator_specs=[GeneratorSpec(generator_enum=Sobol, generator_key_override=None)], transition_criteria=[MinTrials(transition_to='MBM'), MinTrials(transition_to='MBM')], suggested_experiment_status=ExperimentStatus.INITIALIZATION, pausing_criteria=[MaxTrialsAwaitingData(threshold=5)]), GenerationNode(name='MBM', generator_specs=[GeneratorSpec(generator_enum=BoTorch, generator_key_override=None)], transition_criteria=None, suggested_experiment_status=ExperimentStatus.OPTIMIZATION, pausing_criteria=None)]) chosen based on user input and problem structure.
[INFO 10-22 09:18:52] ax.api.client: Generated new trial 10 with parameters {'R': 1462, 'W': 81, 'D': 25, 'P': 0.16247, 'S': 'square', 'O': 'O1'} using GenerationNode Sobol.
[INFO 10-22 09:18:52] ax.api.client: Generated new tr

Trial 10 with parameters {'R': 1462, 'W': 81, 'D': 25, 'P': 0.16247021734714506, 'S': 'square', 'O': 'O1'}
Trial 11 with parameters {'R': 1669, 'W': 65, 'D': 10, 'P': 0.03, 'S': 'round', 'O': 'O2'}
Trial 12 with parameters {'R': 1746, 'W': 82, 'D': 10, 'P': 0.1418944742853875, 'S': 'square', 'O': 'O1'}
Trial 13 with parameters {'R': 1702, 'W': 79, 'D': 30, 'P': 0.03, 'S': 'round', 'O': 'O2'}
Trial 14 with parameters {'R': 1766, 'W': 86, 'D': 10, 'P': 0.03, 'S': 'square', 'O': 'O1'}


,trial_index,R,W,D,S,P,O,visual_quality,sigma_peak,uts
0,0,1700,70,20,round,0.105000,O2,1,127.755842,256.639476
1,1,1462,81,25,square,0.162470,O1,1,212.418007,228.246757
2,2,1733,44,18,round,0.102709,O2,0,124.219927,256.303258
3,3,1997,92,21,round,0.136314,O2,0,172.034853,232.938675
4,4,1649,63,13,square,0.055458,O1,0,131.353171,252.499414
5,5,1462,81,25,square,0.162470,O1,1,203.237882,217.516872
6,6,1687,73,19,round,0.079064,O2,1,137.265759,265.346123
7,7,1687,73,19,round,0.133737,O2,1,128.850675,257.957394
8,8,1435,70,14,round,0.104998,O2,1,211.114702,211.308453
9,9,1718,71,11,round,0.090116,O2,1,117.897305,259.48625


In [86]:
if SAVE_EXCEL:
    exp.to_excel(folder_path / "experiments.xlsx", index=False)